# CareTrace: Rule-Based Triage Agent

This notebook demonstrates the rule-based decision layer of CareTrace.

Key idea:
- Convert structured clinical state into deterministic decisions

Pipeline:
facts → observation predicates → concerns → decision

## Rule Architecture

The rules agent follows a layered structure:

1. Observation Layer  
   Extract signals from structured state  
   (e.g., lethargy, poor intake, no urination)

2. Concern Layer  
   Aggregate signals into clinical concerns  
   (e.g., dehydration, danger red flags)

3. Decision Layer  
   Map concerns to disposition:
   - home_monitor
   - urgent_eval
   - er_now
   - unsupported

# CareTrace: Rules vs LLM Comparison

This notebook compares:

- LLM-based triage (end-to-end reasoning)
- CareTrace rule-based decision system

In [2]:
from src.rules import rules_agent
from llm_openrouter import llm_triage

MODELS = [
    "openai/gpt-4o-mini",
    "anthropic/claude-3-haiku"
]

In [3]:
def extract_decision(output):
    for line in output.split("\n"):
        if line.lower().startswith("decision"):
            return line.split(":")[1].strip()
    return "unknown"


def run_case(name, raw_input, facts):
    print("\n==============================")
    print(name)
    print("==============================")

    print("\nUser Input:")
    print(raw_input.strip())

    print("\n--- LLM Outputs ---")

    for model in MODELS:
        try:
            output = llm_triage(raw_input, model)
            decision = extract_decision(output)

            print(f"\n[{model}]")
            print("Decision:", decision)
            print(output)

        except Exception as e:
            print(f"\n[{model}] ERROR:", e)

    print("\n--- CareTrace Output ---")

    state = {"facts": facts}
    result = rules_agent(state)

    print("Decision:", result["decision"])
    print("Rules triggered:", result["rules_triggered"])

## Scenario 1: Mild Case

Parent: My 6-year-old has a fever, threw up once, and looks really wiped out.
He’s tired but answers me. He’s sipping water, not much though.
He peed earlier this evening.

State:
- Fever
- Vomited once
- Responsive
- Drinking some fluids
- Urinated recently

Expected: **Home monitoring**

In [4]:
scenario_1_input = """
My 6-year-old has a fever, threw up once, and looks really wiped out.
He’s tired but answers me. He’s sipping water, not much though.
He peed earlier this evening.
"""

scenario_1_facts = {
    "fever": "yes",
    "alert": "normal",
    "vomiting": "once",
    "intake": "reduced",
    "urination": "normal"
}

run_case("SCENARIO 1 (MILD)", scenario_1_input, scenario_1_facts)


SCENARIO 1 (MILD)

User Input:
My 6-year-old has a fever, threw up once, and looks really wiped out.
He’s tired but answers me. He’s sipping water, not much though.
He peed earlier this evening.

--- LLM Outputs ---

[openai/gpt-4o-mini]
Decision: urgent_eval
Decision: urgent_eval  
Reason: The child has a fever, has vomited, and appears very tired, which could indicate a more serious underlying condition that requires prompt medical evaluation.

[anthropic/claude-3-haiku]
Decision: urgent_eval
Decision: urgent_eval
Reason: The combination of fever, vomiting, and lethargy in a 6-year-old child warrants an urgent medical evaluation. While the child is responsive and has urinated, the overall presentation suggests a potentially serious underlying condition that requires prompt medical attention.

--- CareTrace Output ---
Decision: urgent_eval
Rules triggered: ['obs:poor_intake', 'obs:fever_present', 'concern:dehydration_concern', 'decision:urgent_eval']


### Observation
The LLM introduces assumptions (e.g., interpreting ‘tired’ as lethargy), which are not explicitly supported by the input. Even when outputs align, LLM reasoning is opaque and may rely on unstated assumptions, while CareTrace provides explicit, verifiable decision logic.

## Scenario 2: Severe Case

Parent: My 6-year-old has a fever, threw up, and looks really wiped out.
He’s barely responding, just lying there. He doesn’t want to drink.
I don’t think he’s peed since this afternoon.

State:
- High fever
- Barely responsive
- Not drinking
- No urination

Expected: **ER immediately**

In [5]:
scenario_2_input = """
My 6-year-old has a fever, threw up, and looks really wiped out.
He’s barely responding, just lying there. He doesn’t want to drink.
I don’t think he’s peed since this afternoon.
"""

scenario_2_facts = {
    "fever": "yes",
    "alert": "reduced",
    "vomiting": "repeated",
    "intake": "none",
    "urination": "none"
}

run_case("SCENARIO 2 (SEVERE)", scenario_2_input, scenario_2_facts)


SCENARIO 2 (SEVERE)

User Input:
My 6-year-old has a fever, threw up, and looks really wiped out.
He’s barely responding, just lying there. He doesn’t want to drink.
I don’t think he’s peed since this afternoon.

--- LLM Outputs ---

[openai/gpt-4o-mini]
Decision: er_now
Decision: er_now  
Reason: The child is exhibiting signs of severe illness, including lethargy, lack of responsiveness, and potential dehydration, which requires immediate medical attention.

[anthropic/claude-3-haiku]
Decision: ER NOW
Decision: ER NOW
Reason: The combination of high fever, vomiting, lethargy, and decreased urine output in a young child is concerning for a serious underlying condition that requires immediate medical evaluation. This child needs prompt emergency care to determine the cause and provide appropriate treatment.

--- CareTrace Output ---
Decision: er_now
Rules triggered: ['obs:lethargy', 'obs:no_urine', 'obs:no_intake', 'obs:repeated_vomiting', 'obs:fever_present', 'concern:danger_red_flag'

### Observation
The LLM inferred additional severity (e.g., assumed high fever when no temperature value was provided), which are not explicitly supported by the input. LLMs may introduce unstated assumptions, while CareTrace only uses explicitly observed signals.

Across both scenarios, LLMs produce plausible decisions, but their reasoning is implicit and may rely on unstated assumptions. CareTrace, in contrast, produces deterministic decisions grounded in explicit, auditable rules.